# Ingestión del archivo `language.csv`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo CSV usando `DataFrameReader` de Spark

In [0]:
from pyspark.sql.types import *
movie_schema = StructType([
    StructField("languageID", IntegerType(), True),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

language_df = (spark.read 
    .schema(movie_schema)
    .option("header", True) 
    .csv(f"{bronze_folder_path}/{v_file_date}/language.csv")
)

## 2. Seleccionar solo las columnas requeridas

In [0]:
from pyspark.sql.functions import col
languages_selected_df = language_df.select(col("languageID"), col("languageName"))

## 3. Cambiar el nombre de las columnas según lo requerido

In [0]:
languages_renamed_df = (languages_selected_df
    .withColumnRenamed("languageID", "language_id")
    .withColumnRenamed("languageName", "language_name")
)


## 4. Agregar la columna `ingestion_date` y `environment` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

languages_final_df = add_ingestion_date(languages_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
languages_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
%sql
SELECT * FROM movie_silver.languages

language_id,language_name,ingestion_date,enviroment,file_date
24574,English,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24575,svenska,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24576,Deutsch,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24577,unknown,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24578,Nihongo,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24579,Français,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24580,Español,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24581,al-?arabiyyah,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24582,Latin,2026-09-13T17:48:28.994502Z,Production,2024-12-16
24583,Khémôrôphéasa,2026-09-13T17:48:28.994502Z,Production,2024-12-16
